In [2]:
import pandas as pd

In [3]:
print('HELLO')

HELLO


In [7]:
import pandas as pd
df = pd.read_csv("../data/games.csv")
df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [13]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================================
# 1. Set project paths
# Assume this notebook is inside: steam project/notebooks/
# =========================================
project_root = Path.cwd().parent
input_path = project_root / "data" / "games.csv"
output_path = project_root / "data cleaned" / "steam_indie_cleaned.csv"

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Output path:", output_path)

# Stop early if file does not exist
if not input_path.exists():
    raise FileNotFoundError(f"Cannot find the file: {input_path}")

# =========================================
# 2. Load raw data
# =========================================
df = pd.read_csv(input_path)

print("\nOriginal shape:", df.shape)
print("\nOriginal columns:")
print(df.columns.tolist())

# =========================================
# 3. Rename columns to cleaner names
# =========================================
rename_dict = {
    "AppID": "app_id",
    "Name": "game_name",
    "Release date": "release_date",
    "Estimated owners": "estimated_owners",
    "Peak CCU": "peak_ccu",
    "Required age": "required_age",
    "Price": "price",
    "DLC count": "dlc_count",
    "About the game": "about_game",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Reviews": "reviews_text",
    "Header image": "header_image",
    "Website": "website",
    "Support url": "support_url",
    "Support email": "support_email",
    "Windows": "windows",
    "Mac": "mac",
    "Linux": "linux",
    "Metacritic score": "metacritic_success_flag",
    "Metacritic url": "metacritic_url",
    "User score": "user_score",
    "Positive": "positive",
    "Negative": "negative",
    "Score rank": "score_rank",
    "Achievements": "achievements",
    "Recommendations": "recommendations",
    "Notes": "notes",
    "Average playtime forever": "avg_playtime_forever",
    "Average playtime two weeks": "avg_playtime_2weeks",
    "Median playtime forever": "median_playtime_forever",
    "Median playtime two weeks": "median_playtime_2weeks",
    "Developers": "developers",
    "Publishers": "publishers",
    "Categories": "categories",
    "Genres": "genres",
    "Tags": "tags",
    "Screenshots": "screenshots",
    "Movies": "movies"
}

df = df.rename(columns=rename_dict)

print("\nRenamed columns:")
print(df.columns.tolist())

# =========================================
# 4. Keep only useful columns for analysis
# =========================================
keep_cols = [
    "app_id",
    "game_name",
    "release_date",
    "estimated_owners",
    "peak_ccu",
    "price",
    "metacritic_success_flag",
    "user_score",
    "positive",
    "negative",
    "recommendations",
    "avg_playtime_forever",
    "avg_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "developers",
    "publishers",
    "categories",
    "genres",
    "tags",
    "windows",
    "mac",
    "linux"
]

missing_cols = [col for col in keep_cols if col not in df.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
    raise KeyError("Some required columns are missing. Please check your CSV column names.")

df = df[keep_cols].copy()

print("\nShape after keeping selected columns:", df.shape)

# =========================================
# 5. Remove duplicate games
# =========================================
df = df.drop_duplicates(subset="app_id")
print("Shape after dropping duplicate app_id:", df.shape)

# =========================================
# 6. Clean text columns
# =========================================
text_cols = ["game_name", "developers", "publishers", "categories", "genres", "tags"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(["nan", "None", ""], np.nan)

# =========================================
# 7. Clean release date and extract year
# =========================================
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

# =========================================
# 8. Clean price
# =========================================
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# If the dataset stores price in cents, convert it
if df["price"].dropna().shape[0] > 0 and df["price"].dropna().median() > 100:
    df["price"] = df["price"] / 100

# Remove negative prices
df.loc[df["price"] < 0, "price"] = np.nan

# Create free-game indicator
df["is_free"] = np.where(df["price"] == 0, 1, 0)

# Create price band
def price_band(x):
    if pd.isna(x):
        return np.nan
    elif x == 0:
        return "Free"
    elif x < 10:
        return "Low"
    elif x < 30:
        return "Mid"
    else:
        return "High"

df["price_band"] = df["price"].apply(price_band)

# =========================================
# 9. Parse estimated owners into numeric midpoint
# Example: "20,000 - 50,000" -> 35000
# =========================================
def parse_owner_midpoint(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").strip()
    nums = re.findall(r"\d+", s)

    if len(nums) >= 2:
        low = int(nums[0])
        high = int(nums[1])
        return (low + high) / 2
    elif len(nums) == 1:
        return float(nums[0])
    else:
        return np.nan

df["owner_midpoint"] = df["estimated_owners"].apply(parse_owner_midpoint)

# Create a simple success band based on owner midpoint
def owner_band(x):
    if pd.isna(x):
        return np.nan
    elif x < 20000:
        return "Low Success"
    elif x < 100000:
        return "Medium Success"
    else:
        return "High Success"

df["success_band"] = df["owner_midpoint"].apply(owner_band)

# =========================================
# 10. Clean Metacritic success flag
# True = successful game
# False = non-successful game
# =========================================
df["metacritic_success_flag"] = (
    df["metacritic_success_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["is_metacritic_positive"] = df["metacritic_success_flag"].map({
    "true": 1,
    "false": 0
})

print("\nMetacritic success flag distribution:")
print(df["is_metacritic_positive"].value_counts(dropna=False))

# =========================================
# 11. Clean numeric columns
# =========================================
numeric_cols = [
    "peak_ccu",
    "user_score",
    "positive",
    "negative",
    "recommendations",
    "avg_playtime_forever",
    "avg_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# =========================================
# 12. Create review-related variables
# =========================================
df["review_count"] = df["positive"].fillna(0) + df["negative"].fillna(0)

df["positive_ratio"] = np.where(
    df["review_count"] > 0,
    df["positive"] / df["review_count"],
    np.nan
)

# =========================================
# 13. Clean platform columns
# Convert true/false to 1/0
# =========================================
for col in ["windows", "mac", "linux"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": 1,
            "false": 0,
            "1": 1,
            "0": 0
        })
    )

# =========================================
# 14. Identify indie games
# Search for 'indie' in genres/categories/tags
# =========================================
df["combined_text"] = (
    df["genres"].fillna("") + " | " +
    df["categories"].fillna("") + " | " +
    df["tags"].fillna("")
).str.lower()

df["is_indie"] = df["combined_text"].str.contains(r"\bindie\b", regex=True, na=False)

df_indie = df[df["is_indie"] == True].copy()

print("\nShape after filtering indie games:", df_indie.shape)

# =========================================
# 15. Drop rows missing key fields
# =========================================
df_indie = df_indie.dropna(subset=["game_name", "owner_midpoint"])

print("Shape after dropping key missing rows:", df_indie.shape)

# =========================================
# 16. Select final columns
# =========================================
final_cols = [
    "app_id",
    "game_name",
    "release_date",
    "release_year",
    "estimated_owners",
    "owner_midpoint",
    "success_band",
    "price",
    "is_free",
    "price_band",
    "peak_ccu",
    "metacritic_success_flag",
    "is_metacritic_positive",
    "user_score",
    "positive",
    "negative",
    "review_count",
    "positive_ratio",
    "recommendations",
    "avg_playtime_forever",
    "avg_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "developers",
    "publishers",
    "categories",
    "genres",
    "tags",
    "windows",
    "mac",
    "linux",
    "is_indie"
]

df_clean = df_indie[final_cols].copy()

# =========================================
# 17. Save cleaned data
# =========================================
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)

# =========================================
# 18. Print summary
# =========================================
print("\nCleaned data saved to:")
print(output_path)

print("\nFinal shape:")
print(df_clean.shape)

print("\nPreview:")
print(df_clean.head())

print("\nTop missing values:")
print(df_clean.isna().sum().sort_values(ascending=False).head(15))

Current working directory: d:\steam project\notebooks
Project root: d:\steam project
Input path: d:\steam project\data\games.csv
Input exists: True
Output path: d:\steam project\data cleaned\steam_indie_cleaned.csv

Original shape: (122611, 39)

Original columns:
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']

Renamed columns:
['app_id', 'game_name', 'release_date', 'estimated_owners', 'peak_ccu', 'required_age', 'price', 'DiscountDLC c

C:\Users\admin\AppData\Local\Temp\ipykernel_1576\1347493297.py:140: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")



Metacritic success flag distribution:
is_metacritic_positive
NaN    121455
Name: count, dtype: int64

Shape after filtering indie games: (82242, 33)
Shape after dropping key missing rows: (82242, 33)

Cleaned data saved to:
d:\steam project\data cleaned\steam_indie_cleaned.csv

Final shape:
(82242, 32)

Preview:
                                 app_id     game_name release_date  \
3292190     버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024          NaT   
700340   Galacatraz: Eject Equip Escape   Feb 1, 2018          NaT   
1157670                     Hepta Beats   May 7, 2021          NaT   
1540330          MUMBA IV: Egypt Jewels  Dec 13, 2021          NaT   
1108640                         OMNIMUS  Sep 25, 2019          NaT   

         release_year  estimated_owners  owner_midpoint success_band  price  \
3292190           NaN                 1             1.0  Low Success    0.0   
700340            NaN                 0             0.0  Low Success    0.0   
1157670           NaN      

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================
# 1. Load cleaned data
# =========================================
project_root = Path.cwd().parent
clean_path = project_root / "data cleaned" / "steam_indie_cleaned.csv"

df = pd.read_csv(clean_path)

print("Data shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nPreview:")
print(df.head())

# =========================================
# 2. Basic overview
# =========================================
print("\n====================")
print("BASIC OVERVIEW")
print("====================")
print("Number of indie games:", len(df))

print("\nMissing values (top 15):")
print(df.isna().sum().sort_values(ascending=False).head(15))

summary_cols = ["owner_midpoint", "price", "release_year", "positive_ratio", "review_count"]
existing_summary_cols = [col for col in summary_cols if col in df.columns]

print("\nSummary statistics:")
print(df[existing_summary_cols].describe())

# =========================================
# 3. Analysis 1: Price vs Success
# =========================================
print("\n====================")
print("ANALYSIS 1: PRICE VS SUCCESS")
print("====================")

price_success = (
    df.groupby("price_band", dropna=False)["owner_midpoint"]
    .agg(["count", "median", "mean"])
    .reset_index()
    .sort_values("median", ascending=False)
)

print("\nOwner-based success by price band:")
print(price_success)

metacritic_by_price = (
    df.groupby("price_band", dropna=False)["is_metacritic_positive"]
    .mean()
    .reset_index()
    .rename(columns={"is_metacritic_positive": "metacritic_success_rate"})
    .sort_values("metacritic_success_rate", ascending=False)
)

print("\nMetacritic success rate by price band:")
print(metacritic_by_price)

# =========================================
# 4. Analysis 2: Release Year vs Success
# =========================================
print("\n====================")
print("ANALYSIS 2: RELEASE YEAR VS SUCCESS")
print("====================")

games_by_year = (
    df.groupby("release_year", dropna=False)["app_id"]
    .count()
    .reset_index()
    .rename(columns={"app_id": "game_count"})
    .sort_values("release_year")
)

print("\nNumber of games by release year:")
print(games_by_year.tail(20))

success_by_year = (
    df.groupby("release_year", dropna=False)["owner_midpoint"]
    .median()
    .reset_index()
    .rename(columns={"owner_midpoint": "median_owner_midpoint"})
    .sort_values("release_year")
)

print("\nMedian owner midpoint by release year:")
print(success_by_year.tail(20))

metacritic_year = (
    df.groupby("release_year", dropna=False)["is_metacritic_positive"]
    .mean()
    .reset_index()
    .rename(columns={"is_metacritic_positive": "metacritic_success_rate"})
    .sort_values("release_year")
)

print("\nMetacritic success rate by release year:")
print(metacritic_year.tail(20))

# =========================================
# 5. Analysis 3: Genre vs Success
# =========================================
print("\n====================")
print("ANALYSIS 3: GENRE VS SUCCESS")
print("====================")

genre_df = df[["app_id", "game_name", "genres", "owner_midpoint", "is_metacritic_positive"]].copy()
genre_df = genre_df.dropna(subset=["genres"])

genre_df["genres"] = genre_df["genres"].astype(str).str.split(",")
genre_df = genre_df.explode("genres")
genre_df["genres"] = genre_df["genres"].str.strip()

genre_counts = (
    genre_df["genres"]
    .value_counts()
    .reset_index()
)
genre_counts.columns = ["genre", "count"]

print("\nTop 15 genres by count:")
print(genre_counts.head(15))

top_genres = genre_counts.head(10)["genre"].tolist()

genre_success = (
    genre_df[genre_df["genres"].isin(top_genres)]
    .groupby("genres")
    .agg(
        game_count=("app_id", "count"),
        median_owner_midpoint=("owner_midpoint", "median"),
        mean_owner_midpoint=("owner_midpoint", "mean"),
        metacritic_success_rate=("is_metacritic_positive", "mean")
    )
    .reset_index()
    .sort_values("median_owner_midpoint", ascending=False)
)

print("\nSuccess metrics for top 10 genres:")
print(genre_success)

# =========================================
# 6. Analysis 4: Review Positivity vs Success
# =========================================
print("\n====================")
print("ANALYSIS 4: REVIEW POSITIVITY VS SUCCESS")
print("====================")

def review_band(x):
    if pd.isna(x):
        return np.nan
    elif x < 0.6:
        return "Low"
    elif x < 0.8:
        return "Medium"
    else:
        return "High"

df["review_band"] = df["positive_ratio"].apply(review_band)

review_success = (
    df.groupby("review_band", dropna=False)["owner_midpoint"]
    .agg(["count", "median", "mean"])
    .reset_index()
    .sort_values("median", ascending=False)
)

print("\nOwner-based success by review positivity band:")
print(review_success)

review_meta = (
    df.groupby("review_band", dropna=False)["is_metacritic_positive"]
    .mean()
    .reset_index()
    .rename(columns={"is_metacritic_positive": "metacritic_success_rate"})
    .sort_values("metacritic_success_rate", ascending=False)
)

print("\nMetacritic success rate by review positivity band:")
print(review_meta)

# =========================================
# 7. Analysis 5: Simple multi-factor summary
# =========================================
print("\n====================")
print("ANALYSIS 5: MULTI-FACTOR SUMMARY")
print("====================")

multi_summary = (
    df.groupby(["price_band", "review_band"], dropna=False)
    .agg(
        game_count=("app_id", "count"),
        median_owner_midpoint=("owner_midpoint", "median"),
        mean_owner_midpoint=("owner_midpoint", "mean"),
        metacritic_success_rate=("is_metacritic_positive", "mean")
    )
    .reset_index()
    .sort_values(["price_band", "review_band"])
)

print("\nPrice band + review band summary:")
print(multi_summary)

# =========================================
# 8. Analysis 6: Correlation table
# =========================================
print("\n====================")
print("ANALYSIS 6: CORRELATION TABLE")
print("====================")

analysis_cols = [
    "owner_midpoint",
    "price",
    "release_year",
    "peak_ccu",
    "user_score",
    "positive",
    "negative",
    "review_count",
    "positive_ratio",
    "recommendations",
    "avg_playtime_forever",
    "median_playtime_forever",
    "is_metacritic_positive"
]

existing_analysis_cols = [col for col in analysis_cols if col in df.columns]

corr_df = df[existing_analysis_cols].corr(numeric_only=True)

print("\nCorrelation matrix:")
print(corr_df)

# =========================================
# 9. Optional: save results to outputs folder
# =========================================
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

price_success.to_csv(output_dir / "price_success_summary.csv", index=False)
metacritic_by_price.to_csv(output_dir / "price_metacritic_summary.csv", index=False)
games_by_year.to_csv(output_dir / "games_by_year.csv", index=False)
success_by_year.to_csv(output_dir / "success_by_year.csv", index=False)
metacritic_year.to_csv(output_dir / "metacritic_by_year.csv", index=False)
genre_counts.to_csv(output_dir / "genre_counts.csv", index=False)
genre_success.to_csv(output_dir / "genre_success_summary.csv", index=False)
review_success.to_csv(output_dir / "review_success_summary.csv", index=False)
review_meta.to_csv(output_dir / "review_metacritic_summary.csv", index=False)
multi_summary.to_csv(output_dir / "multi_factor_summary.csv", index=False)
corr_df.to_csv(output_dir / "correlation_matrix.csv")

print("\nAll summary tables saved to:")
print(output_dir)

Data shape: (82242, 32)

Columns:
['app_id', 'game_name', 'release_date', 'release_year', 'estimated_owners', 'owner_midpoint', 'success_band', 'price', 'is_free', 'price_band', 'peak_ccu', 'metacritic_success_flag', 'is_metacritic_positive', 'user_score', 'positive', 'negative', 'review_count', 'positive_ratio', 'recommendations', 'avg_playtime_forever', 'avg_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'windows', 'mac', 'linux', 'is_indie']

Preview:
                           app_id     game_name  release_date  release_year  \
0     버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024           NaN           NaN   
1  Galacatraz: Eject Equip Escape   Feb 1, 2018           NaN           NaN   
2                     Hepta Beats   May 7, 2021           NaN           NaN   
3          MUMBA IV: Egypt Jewels  Dec 13, 2021           NaN           NaN   
4                         OMNIMUS  Sep 25, 2019           NaN 